In [ ]:
from dotenv import load_dotenv
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

In [32]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [33]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [ ]:
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# from langchain_openai import OpenAIEmbeddings
# embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

In [34]:
embeddings.embed_query("hello world")

[-0.024183841,
 0.0098769935,
 0.0074856607,
 -0.06730222,
 6.1710525e-05,
 0.0056642424,
 -0.015386985,
 0.014830949,
 0.009587027,
 0.009637222,
 -0.0069075353,
 -0.015822116,
 0.031628944,
 0.048462078,
 0.10820047,
 0.011083316,
 -0.004259401,
 -0.013756547,
 0.0072839246,
 -0.010003761,
 0.005551397,
 -0.00973027,
 0.022045601,
 0.01386423,
 -0.034425125,
 0.011548195,
 0.02382501,
 0.0041801715,
 0.029232847,
 0.017636301,
 0.016677855,
 -0.010255939,
 -0.025482079,
 0.009086535,
 -0.00635536,
 0.016052574,
 0.011485045,
 0.0018753876,
 -0.011397894,
 -0.011110411,
 -0.004131313,
 0.010747639,
 -0.01147486,
 -0.012512612,
 -0.020175282,
 -0.015779916,
 -0.014151312,
 -0.0122082215,
 -0.0042187893,
 0.015479805,
 -0.028816793,
 -0.009396702,
 -0.005720598,
 -0.15016583,
 -0.010830634,
 0.0054414957,
 -0.012195118,
 0.027744047,
 -0.012230321,
 -0.018721882,
 -0.007321619,
 0.0042207795,
 -0.014443484,
 -0.006819255,
 0.0025285373,
 -0.022827905,
 -0.012578101,
 0.01398951,
 -0.025

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
documents=["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."]

In [37]:
my_query  = "Who is a prime minister of India?"

In [38]:
embedded_docs = embeddings.embed_documents(documents)
embedded_query = embeddings.embed_query(my_query)

In [39]:
# calculate weights and cosine will take highst priority
cosine_similarity([embedded_query],embedded_docs)

array([[0.55572604, 0.61247309, 0.77749546]])

In [40]:
# will check euclidean_distances for wights checking this 
# This euclidean_distances will take least priority 
from sklearn.metrics.pairwise import euclidean_distances

In [41]:
euclidean_distances([embedded_query], embedded_docs)

array([[0.9426282 , 0.88037141, 0.66709004]])

In [42]:
# Lets create Faiss database

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

myemd = embeddings.embed_query('hello')
len(myemd)

3072

In [43]:
# Create Index 

index = faiss.IndexFlatL2(3072)

In [44]:
# This is my vector  store
#  inside this vectorstore the data  is being stor insise the index
#  As of now it is inmomory but we can also store it in disk as well

vector_store=FAISS(embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={}
        )

In [45]:
# Add some text to vector db
vector_store.add_texts(
    ["Washington is the capital of USA",
    "Donald Trump is a president of USA",
    "Narendra Modi is a prime minister of India."])
    

['cfecd1d6-839a-4ef6-86da-cff7e14ffffd',
 '029e17cf-30fb-4964-8098-163cff3d756c',
 'a0a516f9-d7c2-4e4d-9997-573c982ae3cc']

In [47]:
# get the each index of document 
vector_store.index_to_docstore_id

{0: 'cfecd1d6-839a-4ef6-86da-cff7e14ffffd',
 1: '029e17cf-30fb-4964-8098-163cff3d756c',
 2: 'a0a516f9-d7c2-4e4d-9997-573c982ae3cc'}

In [48]:
# Lets create any index , now lets create index of 1    
faiss_index_id =1

In [49]:
# using above index=1 we can pass into vector db and get the document id
docstore_id = vector_store.index_to_docstore_id[faiss_index_id]

In [50]:
# Lets search doc_id in vector db
vector_store.docstore.search(docstore_id)

Document(id='029e17cf-30fb-4964-8098-163cff3d756c', metadata={}, page_content='Donald Trump is a president of USA')

In [51]:
#by K  select top k result  from indexs 
vector_store.similarity_search("Who is a president of india?", k=1)

[Document(id='a0a516f9-d7c2-4e4d-9997-573c982ae3cc', metadata={}, page_content='Narendra Modi is a prime minister of India.')]

In [52]:
vector_store.similarity_search('Who is a president of USA?',k=1)

[Document(id='029e17cf-30fb-4964-8098-163cff3d756c', metadata={}, page_content='Donald Trump is a president of USA')]

In [53]:
vector_store.similarity_search('Who is a president of USA?', k=2)

[Document(id='029e17cf-30fb-4964-8098-163cff3d756c', metadata={}, page_content='Donald Trump is a president of USA'),
 Document(id='cfecd1d6-839a-4ef6-86da-cff7e14ffffd', metadata={}, page_content='Washington is the capital of USA')]